# Experiment 22 - Identity Encoding V2

This experiment builds on the strong Experiment 21G result.

The goal is to test leakage-safe out-of-fold target encoding for exact numerical values, add frequency information, and test selected exact-value numerical combinations.

The original numerical features and one-hot categorical features are retained.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score

from xgboost import XGBClassifier

PROJECT_ROOT = r'C:\Users\aakif\Documents\DataCompetition'
TRAIN_PATH = PROJECT_ROOT + r'\data\train.csv'

train = pd.read_csv(TRAIN_PATH)

X = train.drop(columns=['Will_Buy_EV', 'id']).copy()
y = train['Will_Buy_EV'].map({'No': 0, 'Yes': 1}).astype(int)

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

numeric_cols = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=['number']).columns.tolist()

print('Training rows:', len(X_train))
print('Validation rows:', len(X_valid))
print('Numeric columns:', numeric_cols)
print('Categorical columns:', categorical_cols)

Training rows: 534932
Validation rows: 133733
Numeric columns: ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level']
Categorical columns: ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']


In [2]:
def make_identity_key(series):
    return series.astype('string').fillna('__MISSING__')


def fit_mapping(values, target, smoothing=20):
    temp = pd.DataFrame({
        'value': values,
        'target': target.to_numpy()
    })

    global_mean = float(target.mean())
    stats = temp.groupby('value', dropna=False)['target'].agg(['mean', 'count'])
    smoothed = (
        stats['count'] * stats['mean'] + smoothing * global_mean
    ) / (stats['count'] + smoothing)

    return smoothed.to_dict(), global_mean


def apply_mapping(values, mapping, global_mean):
    return values.map(mapping).fillna(global_mean).astype(float)


def add_oof_identity_features(X_fit, y_fit, X_apply, columns, n_splits=3, smoothing=20, add_frequency=False):
    X_fit = X_fit.copy()
    X_apply = X_apply.copy()

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    for col in columns:
        fit_keys = make_identity_key(X_fit[col])
        apply_keys = make_identity_key(X_apply[col])

        oof_values = np.zeros(len(X_fit), dtype=float)

        for train_idx, fold_idx in skf.split(X_fit, y_fit):
            fold_keys = fit_keys.iloc[train_idx]
            fold_target = y_fit.iloc[train_idx]

            mapping, global_mean = fit_mapping(
                fold_keys,
                fold_target,
                smoothing=smoothing
            )

            oof_values[fold_idx] = apply_mapping(
                fit_keys.iloc[fold_idx],
                mapping,
                global_mean
            ).to_numpy()

        full_mapping, full_global_mean = fit_mapping(
            fit_keys,
            y_fit,
            smoothing=smoothing
        )

        X_fit[f'{col}__identity_target'] = oof_values
        X_apply[f'{col}__identity_target'] = apply_mapping(
            apply_keys,
            full_mapping,
            full_global_mean
        ).to_numpy()

        if add_frequency:
            frequencies = fit_keys.value_counts(dropna=False)
            X_fit[f'{col}__identity_frequency'] = fit_keys.map(frequencies).fillna(0).astype(float).to_numpy()
            X_apply[f'{col}__identity_frequency'] = apply_keys.map(frequencies).fillna(0).astype(float).to_numpy()

    return X_fit, X_apply


def add_pair_identity_target(X_fit, y_fit, X_apply, pairs, n_splits=3, smoothing=20):
    X_fit = X_fit.copy()
    X_apply = X_apply.copy()

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    for left, right in pairs:
        feature_name = f'{left}__x__{right}__identity_target'

        fit_keys = (
            make_identity_key(X_fit[left]) + '||' + make_identity_key(X_fit[right])
        )
        apply_keys = (
            make_identity_key(X_apply[left]) + '||' + make_identity_key(X_apply[right])
        )

        oof_values = np.zeros(len(X_fit), dtype=float)

        for train_idx, fold_idx in skf.split(X_fit, y_fit):
            mapping, global_mean = fit_mapping(
                fit_keys.iloc[train_idx],
                y_fit.iloc[train_idx],
                smoothing=smoothing
            )

            oof_values[fold_idx] = apply_mapping(
                fit_keys.iloc[fold_idx],
                mapping,
                global_mean
            ).to_numpy()

        full_mapping, full_global_mean = fit_mapping(
            fit_keys,
            y_fit,
            smoothing=smoothing
        )

        X_fit[feature_name] = oof_values
        X_apply[feature_name] = apply_mapping(
            apply_keys,
            full_mapping,
            full_global_mean
        ).to_numpy()

    return X_fit, X_apply

In [3]:
def build_preprocessor(X_frame):
    numeric = X_frame.select_dtypes(include=['number']).columns.tolist()
    categorical = X_frame.select_dtypes(exclude=['number']).columns.tolist()

    numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median'))
    ])

    categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])

    return ColumnTransformer([
        ('num', numeric_pipeline, numeric),
        ('cat', categorical_pipeline, categorical)
    ])


def run_xgb(X_tr, y_tr, X_va, y_va, label):
    preprocessor = build_preprocessor(X_tr)

    model = XGBClassifier(
        n_estimators=800,
        max_depth=5,
        learning_rate=0.04,
        min_child_weight=2,
        subsample=0.90,
        colsample_bytree=0.85,
        gamma=0,
        reg_alpha=0,
        reg_lambda=1,
        objective='binary:logistic',
        eval_metric='auc',
        tree_method='hist',
        random_state=42,
        n_jobs=-1
    )

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    pipeline.fit(X_tr, y_tr)
    predictions = pipeline.predict_proba(X_va)[:, 1]
    score = roc_auc_score(y_va, predictions)

    print(f'{label}: {score:.6f}')
    return score


In [4]:
results = []

pair_columns = [
    ('Age', 'Annual_Income_USD'),
    ('Annual_Income_USD', 'Daily_Commute_km'),
    ('Daily_Commute_km', 'Charging_Stations_Near_Work'),
    ('Charging_Stations_Near_Home', 'Charging_Stations_Near_Work'),
    ('Number_of_Cars_Owned', 'Annual_Income_USD')
]

print('Building leakage-safe exact-value target encodings...')

X21G_train, X21G_valid = add_oof_identity_features(
    X_train,
    y_train,
    X_valid,
    numeric_cols,
    n_splits=3,
    smoothing=20,
    add_frequency=False
)

score = run_xgb(
    X21G_train,
    y_train,
    X21G_valid,
    y_valid,
    '22A_OOF_Identity_Target'
)
results.append(('22A_OOF_Identity_Target', score, X21G_train.shape[1]))

print('')
print('Adding identity frequency features...')

X_freq_train, X_freq_valid = add_oof_identity_features(
    X_train,
    y_train,
    X_valid,
    numeric_cols,
    n_splits=3,
    smoothing=20,
    add_frequency=True
)

score = run_xgb(
    X_freq_train,
    y_train,
    X_freq_valid,
    y_valid,
    '22B_OOF_Identity_Target_Frequency'
)
results.append(('22B_OOF_Identity_Target_Frequency', score, X_freq_train.shape[1]))

print('')
print('Adding selected exact-value pair target encodings...')

X_pair_train, X_pair_valid = add_oof_identity_features(
    X_train,
    y_train,
    X_valid,
    numeric_cols,
    n_splits=3,
    smoothing=20,
    add_frequency=True
)

X_pair_train, X_pair_valid = add_pair_identity_target(
    X_pair_train,
    y_train,
    X_pair_valid,
    pair_columns,
    n_splits=3,
    smoothing=20
)

score = run_xgb(
    X_pair_train,
    y_train,
    X_pair_valid,
    y_valid,
    '22C_OOF_Identity_Frequency_Pairs'
)
results.append(('22C_OOF_Identity_Frequency_Pairs', score, X_pair_train.shape[1]))

results_df = pd.DataFrame(results, columns=['Experiment', 'ROC_AUC', 'Feature_Count'])
results_df = results_df.sort_values('ROC_AUC', ascending=False).reset_index(drop=True)

print('')
print('============================================================')
print('EXPERIMENT 22 RESULTS')
print('============================================================')
print(results_df.to_string(index=False))

previous_best = 0.941815
best_score = float(results_df.iloc[0]['ROC_AUC'])
print('')
print(f'Previous local best: {previous_best:.6f}')
print(f'Best Experiment 22 score: {best_score:.6f}')
print(f'Difference vs previous best: {best_score - previous_best:+.6f}')

Building leakage-safe exact-value target encodings...
22A_OOF_Identity_Target: 0.944838

Adding identity frequency features...
22B_OOF_Identity_Target_Frequency: 0.945243

Adding selected exact-value pair target encodings...
22C_OOF_Identity_Frequency_Pairs: 0.945236

EXPERIMENT 22 RESULTS
                       Experiment  ROC_AUC  Feature_Count
22B_OOF_Identity_Target_Frequency 0.945243             27
 22C_OOF_Identity_Frequency_Pairs 0.945236             32
          22A_OOF_Identity_Target 0.944838             20

Previous local best: 0.941815
Best Experiment 22 score: 0.945243
Difference vs previous best: +0.003428
